# M19 · Hessienne, conditionnement et convergence — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m19_hessienne_convergence/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m19_hessienne_convergence/03_notebook.ipynb`, créée par `python tools/start_chapter.py M19`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M19.1 à M19.9, M19.14, M19.15 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M19.19 | La hessienne par différences centrées · `numerical_hessian` | 🔨 | ★★ | 35 |
|  | M19.20 | Le pas critique d'une matrice symétrique définie positive, et les itérés de Richardson · `critical_step_size, richardson_iterates` | 🔨 | ★★ | 30 |
|  | M19.21 | Une récurrence linéaire d'ordre d · `linear_recurrence` | 🔨 | ★ | 15 |
| B | M19.22 | `scipy.sparse.linalg.cg` : le gradient conjugué contre Richardson | 📦 | ★ | 15 |
|  | M19.23 | Prédire le nombre d'itérations de Richardson selon ω, puis vérifier | 🔮 | ★ | 10 |
|  | M19.24 | Le nombre d'itérations contre le conditionnement : une droite en échelle logarithmique | 🔬 | ★★ | 20 |
| C | M19.25 | Les itérés de Richardson sur les lignes de niveau d'une forme quadratique | 🎨 | ★ | 20 |
|  | M19.26 | Chasse à l'erreur : un pas réglé sur λmin, une hessienne non symétrique, le module d'une racine complexe | 🐛 | ★ | 25 |
|  | M19.27 | Un critère d'arrêt : le résidu relatif ‖b − A x‖/‖b‖ et un nombre maximal d'itérations | 🛠️ | ★ | 15 |
|  | M19.28 | Défi : le schéma saute-mouton d'un oscillateur, x_{k+1} = (2 − h²Ω²) x_k − x_{k−1}, borné si et seulement si hΩ < 2 | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M19")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : une hessienne et Taylor d'ordre 2, des points critiques en dimension 3, une itération linéaire, Richardson à la main, le meilleur pas et le nombre d'itérations, deux récurrences d'ordre 2, un gradient estimé sur un échantillon, deux projections, et deux ponts.
- Écrire les quatre fonctions de `mymath/iterative.py` (`numerical_hessian`, `critical_step_size`, `richardson_iterates`, `linear_recurrence`) et les tester.
- Comparer le gradient conjugué de SciPy à Richardson, prédire puis compter des itérations, mesurer leur croissance avec le conditionnement, dessiner des itérés sur des lignes de niveau, corriger trois erreurs classiques, écrire un critère d'arrêt, et relever un défi : la stabilité du schéma saute-mouton.

**Rappel express.** Hessienne $H_{ij} = \frac{\partial^2 f}{\partial x_i\partial x_j}$, symétrique ; $f(\mathbf{x} + \mathbf{h}) \approx f(\mathbf{x}) + \nabla f^\top\mathbf{h} + \frac12\mathbf{h}^\top\mathbf{H}\mathbf{h}$ ; par différences centrées, $h \approx 10^{-4}$. En un point critique : valeurs propres toutes $> 0$, minimum ; toutes $< 0$, maximum ; des deux signes, selle. $\mathbf{x}_{k+1} = \mathbf{M}\mathbf{x}_k + \mathbf{c}$, $\mathbf{M}$ symétrique : convergence depuis tout départ si et seulement si $\rho = \max_i\lvert\lambda_i\rvert < 1$ ; $k \ge \frac{d\ln 10}{-\ln\rho}$ pour gagner $d$ décimales. Richardson $\mathbf{x} \leftarrow \mathbf{x} + \omega(\mathbf{b} - \mathbf{A}\mathbf{x})$ : $0 < \omega < \frac{2}{\lambda_{\max}}$, $\rho(\omega) = \max(\lvert 1 - \omega\lambda_{\min}\rvert, \lvert 1 - \omega\lambda_{\max}\rvert)$, meilleur pas $\frac{2}{\lambda_{\min} + \lambda_{\max}}$ de facteur $\frac{\kappa - 1}{\kappa + 1}$, environ $1{,}15\,\kappa\,d$ itérations ; gradient conjugué : de l'ordre de $\sqrt\kappa$, au plus $n$. Récurrence $u_{k+2} = p\,u_{k+1} - q\,u_k$ : racines de $r^2 - pr + q$, complexes de module $\sqrt q$ si $p^2 < 4q$ ; $u_k \to 0$ depuis tout départ si et seulement si $\lvert q\rvert < 1$ et $\lvert p\rvert < 1 + q$.

## Partie 0 · Vérifier tes exercices papier (✏️ M19.1 à M19.9, 🔗 M19.14 et M19.15)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (une valeur exacte se tape aussi comme un calcul, `1/8`) ;
- une liste `[0.2, -1]`, ou une liste de listes ;
- une matrice comme la liste de ses **lignes** : `[[1, 0], [-2, 3]]` ; un nombre décimal s'y écrit avec un point, `[[0.25, 1], [0, 2]]` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- une lettre entre guillemets pour un choix, `"D"` ; `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M19.1 — La hessienne d'une fonction de trois variables, et son développement de Taylor d'ordre 2**

In [ ]:
# Ex M19.1: the values you found on paper
answer_M19_1a = ...  # a) a list of 3 numbers
answer_M19_1b = ...  # b) a matrix (list of rows)
answer_M19_1c = ...  # c) a number
answer_M19_1d = ...  # d) a number
answer_M19_1e = ...  # e) a number (3 decimals)
answer_M19_1f = ...  # f) a number
answer_M19_1g = ...  # g) a number
answer_M19_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M19_1a, answer_M19_1b, answer_M19_1c, answer_M19_1d, answer_M19_1e, answer_M19_1f, answer_M19_1g, answer_M19_1h]):
    wbm.check(f"M19.1{letter}", answer)

**Ex M19.2 — Classer les points critiques d'une fonction de trois variables**

In [ ]:
# Ex M19.2: the values you found on paper
answer_M19_2a = ...  # a) an integer
answer_M19_2b = ...  # b) a list of 4 lists of 3 numbers, by increasing x
answer_M19_2c = ...  # c) a list of 3 numbers, in increasing order
answer_M19_2d = ...  # d) a list of 3 numbers, in increasing order
answer_M19_2e = ...  # e) a list of 4 integers
answer_M19_2f = ...  # f) a number
answer_M19_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M19_2a, answer_M19_2b, answer_M19_2c, answer_M19_2d, answer_M19_2e, answer_M19_2f, answer_M19_2g]):
    wbm.check(f"M19.2{letter}", answer)

**Ex M19.3 — Une itération linéaire 2 × 2 : point fixe, erreur, et convergence par les valeurs propres**

In [ ]:
# Ex M19.3: the values you found on paper
answer_M19_3a = ...  # a) a list of 2 numbers
answer_M19_3b = ...  # b) a list of 2 lists of 2 numbers: [x1, x2]
answer_M19_3c = ...  # c) a list of 2 numbers
answer_M19_3d = ...  # d) a list of 2 numbers: [a, b]
answer_M19_3e = ...  # e) a list of 2 numbers
answer_M19_3f = ...  # f) an integer
answer_M19_3g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M19_3a, answer_M19_3b, answer_M19_3c, answer_M19_3d, answer_M19_3e, answer_M19_3f, answer_M19_3g]):
    wbm.check(f"M19.3{letter}", answer)

**Ex M19.4 — Résoudre un système 2 × 2 par l'itération de Richardson : trois pas à la main, et le pas critique**

In [ ]:
# Ex M19.4: the values you found on paper
answer_M19_4a = ...  # a) a list of 2 numbers
answer_M19_4b = ...  # b) a list of 3 lists of 2 numbers (6 decimals): [x1, x2, x3]
answer_M19_4c = ...  # c) a list of 2 numbers (6 decimals)
answer_M19_4d = ...  # d) a list of 2 numbers, in decreasing order
answer_M19_4e = ...  # e) a list of 2 numbers
answer_M19_4f = ...  # f) a number
answer_M19_4g = ...  # g) a number
answer_M19_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M19_4a, answer_M19_4b, answer_M19_4c, answer_M19_4d, answer_M19_4e, answer_M19_4f, answer_M19_4g, answer_M19_4h]):
    wbm.check(f"M19.4{letter}", answer)

**Ex M19.5 — Le meilleur pas de Richardson (énoncé), et le nombre d'itérations pour gagner six décimales**

In [ ]:
# Ex M19.5: the values you found on paper
answer_M19_5a = ...  # a) a list of 3 numbers, in increasing order
answer_M19_5b = ...  # b) an irreducible fraction, as text
answer_M19_5c = ...  # c) an irreducible fraction, as text
answer_M19_5d = ...  # d) an irreducible fraction, as text
answer_M19_5e = ...  # e) an integer
answer_M19_5f = ...  # f) an irreducible fraction, as text
answer_M19_5g = ...  # g) an integer
answer_M19_5h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M19_5a, answer_M19_5b, answer_M19_5c, answer_M19_5d, answer_M19_5e, answer_M19_5f, answer_M19_5g, answer_M19_5h]):
    wbm.check(f"M19.5{letter}", answer)

**Ex M19.6 — Une récurrence d'ordre 2 à racines réelles : terme général et convergence**

In [ ]:
# Ex M19.6: the values you found on paper
answer_M19_6a = ...  # a) a list of 2 numbers, in decreasing order
answer_M19_6b = ...  # b) a list of 2 numbers: [alpha, beta]
answer_M19_6c = ...  # c) a list of 2 numbers: [u2, u3]
answer_M19_6d = ...  # d) a number (6 decimals)
answer_M19_6e = ...  # e) a list of 2 numbers, in decreasing order
answer_M19_6f = ...  # f) a number
answer_M19_6g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M19_6a, answer_M19_6b, answer_M19_6c, answer_M19_6d, answer_M19_6e, answer_M19_6f, answer_M19_6g]):
    wbm.check(f"M19.6{letter}", answer)

**Ex M19.7 — Une récurrence d'ordre 2 à racines complexes : module et oscillation amortie**

In [ ]:
# Ex M19.7: the values you found on paper
answer_M19_7a = ...  # a) a number
answer_M19_7b = ...  # b) a list of 2 numbers: [real part, positive imaginary part]
answer_M19_7c = ...  # c) a number
answer_M19_7d = ...  # d) a number (4 decimals)
answer_M19_7e = ...  # e) a list of 4 numbers (6 decimals): [u2, u3, u4, u5]
answer_M19_7f = ...  # f) a list of 2 numbers: [A, B]
answer_M19_7g = ...  # g) an integer
answer_M19_7h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_M19_7a, answer_M19_7b, answer_M19_7c, answer_M19_7d, answer_M19_7e, answer_M19_7f, answer_M19_7g, answer_M19_7h]):
    wbm.check(f"M19.7{letter}", answer)

**Ex M19.8 — Une pente estimée sur B points tirés au hasard : espérance et variance de l'estimation**

In [ ]:
# Ex M19.8: the values you found on paper
answer_M19_8a = ...  # a) a number
answer_M19_8b = ...  # b) a list of 4 numbers
answer_M19_8c = ...  # c) a number
answer_M19_8d = ...  # d) a number
answer_M19_8e = ...  # e) a number (4 decimals)
answer_M19_8f = ...  # f) an integer
answer_M19_8g = ...  # g) a number (4 decimals)
answer_M19_8h = ...  # h) an irreducible fraction, as text
answer_M19_8i = ...  # i) an irreducible fraction, as text

for letter, answer in zip("abcdefghi", [answer_M19_8a, answer_M19_8b, answer_M19_8c, answer_M19_8d, answer_M19_8e, answer_M19_8f, answer_M19_8g, answer_M19_8h, answer_M19_8i]):
    wbm.check(f"M19.8{letter}", answer)

**Ex M19.9 — Projeter un même vecteur sur une boule et sur un pavé : deux points, deux directions**

In [ ]:
# Ex M19.9: the values you found on paper
answer_M19_9a = ...  # a) a number
answer_M19_9b = ...  # b) a list of 3 numbers
answer_M19_9c = ...  # c) a list of 3 numbers
answer_M19_9d = ...  # d) a list of 2 numbers (4 decimals)
answer_M19_9e = ...  # e) a number (4 decimals)
answer_M19_9f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_M19_9a, answer_M19_9b, answer_M19_9c, answer_M19_9d, answer_M19_9e, answer_M19_9f]):
    wbm.check(f"M19.9{letter}", answer)

**Ex M19.14 — Pont : un pas de Richardson est un pas de descente sur ½ xᵀA x − bᵀx (énoncé, sans calcul)**

In [ ]:
# Ex M19.14: the values you found on paper
answer_M19_14a = ...  # a) "A", "B", "C" or "D"
answer_M19_14b = ...  # b) "A", "B", "C" or "D"
answer_M19_14c = ...  # c) "A", "B", "C" or "D"
answer_M19_14d = ...  # d) "A", "B", "C" or "D"
answer_M19_14e = ...  # e) "A", "B", "C" or "D"
answer_M19_14f = ...  # f) "A", "B", "C" or "D"
answer_M19_14g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M19_14a, answer_M19_14b, answer_M19_14c, answer_M19_14d, answer_M19_14e, answer_M19_14f, answer_M19_14g]):
    wbm.check(f"M19.14{letter}", answer)

**Ex M19.15 — Pont : le préconditionnement diagonal, une idée commune à plusieurs méthodes de workbookIA 19**

In [ ]:
# Ex M19.15: the values you found on paper
answer_M19_15a = ...  # a) a list of 2 numbers (3 decimals), in decreasing order
answer_M19_15b = ...  # b) a number (1 decimal)
answer_M19_15c = ...  # c) a matrix (list of rows)
answer_M19_15d = ...  # d) a number
answer_M19_15e = ...  # e) a list of 2 integers
answer_M19_15f = ...  # f) a list of 2 numbers

for letter, answer in zip("abcdef", [answer_M19_15a, answer_M19_15b, answer_M19_15c, answer_M19_15d, answer_M19_15e, answer_M19_15f]):
    wbm.check(f"M19.15{letter}", answer)

## Partie A · Ta librairie : la hessienne numérique, le pas critique, les itérés de Richardson et les récurrences

Fiche §1, §2, §4, §5 et §6. Les quatre fonctions de `mymath/iterative.py`, leurs tests, et un usage de chacune.

In [ ]:
# Tools of the notebook (parts A to C)
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse
import scipy.sparse.linalg
import sympy
from matplotlib.contour import ContourSet

from wbm.nbtools import StopCheck, checked, drawn, filled, run_pytest_group, verdict

TEST_M19 = "tests/test_m19_iterative.py"          # the tests of mymath/iterative.py

### Ex M19.19 — La hessienne par différences centrées · `numerical_hessian` 🔨 ★★ ⏱️ 35 min
**Objectif :** programmer la hessienne d'une fonction de $n$ variables par différences centrées, choisir son pas, et classer un point critique de $\mathbb{R}^3$ par les signes de ses valeurs propres.  
**Prérequis :** fiche §1 et §2 ; M05a (le pas des différences finies) · **mymath :** `iterative.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/iterative.py`, lis la docstring du module puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. ⚠️ Le squelette n'importe rien : ajoute `import numpy as np` en tête du fichier (et `import numbers` si tu t'en sers pour reconnaître un entier). `np.linalg.eigvalsh` est permis dans `critical_step_size` seulement ; ni `np.linalg.solve`, ni `np.linalg.inv`, ni `scipy.optimize`, ni `scipy.sparse.linalg` : les tests comparent tes résultats aux leurs, et un test lit ton module. Les nombres se rendent en `float` de Python, les vecteurs et les matrices en tableaux NumPy de flottants ; ne modifie jamais les tableaux reçus.

Écris `numerical_hessian(f, x, h=1e-4)` (fiche §1), en suivant sa docstring : la hessienne $(n, n)$ de $f : \mathbb{R}^n \to \mathbb{R}$ en $\mathbf{x}$, par la dérivée seconde centrée sur la diagonale et par la formule à quatre points hors de la diagonale, toutes deux avec le pas `h` ; le résultat est **exactement** symétrique ; `ValueError` si `x` n'est pas un tableau 1-D non vide, ou si $h \le 0$ ; `x` n'est pas modifié.

**L'usage.** La cellule fournie donne $g(x, y) = \frac{x^2}{1 + y^2} + \cos(xy)$ (`g_19` pour NumPy, `G_EXPR_19` pour SymPy), le point $\mathbf{p} = (1{,}2 ;\ -0{,}5)$ (`P_19`), la fonction $s(x, y, z) = \cosh x + \cos y + xz + z^2$ (`s_19`), dont l'origine est un point critique, et les sept pas $h = 10^{-2}, 10^{-3}, \ldots, 10^{-8}$ (`STEPS_19`). Avec ta fonction, calcule :
- a) la hessienne numérique de $g$ en $\mathbf{p}$ (une matrice, 4 décimales) ;
- b) les valeurs propres de la hessienne numérique de $s$ en $\mathbf{0}$, en ordre croissant (`np.linalg.eigvalsh` ; une liste de 3 nombres, 4 décimales) ;
- c) l'indice de ce point critique, le nombre de ses valeurs propres négatives (un entier) ;
- d) sa nature : (A) minimum local ; (B) maximum local ; (C) point selle ; (D) le test ne conclut pas (une lettre entre guillemets) ;
- e) la hessienne exacte de $g$ en $\mathbf{p}$ par SymPy (`sympy.hessian`, puis `.subs` et `np.array(..., dtype=float)`), puis, pour chacun des sept pas de `STEPS_19`, le plus grand écart $\max_{ij}\lvert H^{\text{num}}_{ij} - H_{ij}\rvert$ : l'entier $k$ du meilleur pas $h = 10^{-k}$.

Dans ta copie : vérifie que $\mathbf{0}$ est un point critique de $s$, écris sa hessienne exacte et retrouve b) (un bloc $2 \times 2$, et un $-1$). Dans quelle direction $s$ descend-elle depuis $\mathbf{0}$ ? D'où viennent les grandes erreurs aux deux bouts de e) ? Pourquoi le meilleur pas d'une dérivée seconde est-il plus grand que celui d'une dérivée première (M05a) ? Combien de fois ta fonction évalue-t-elle $f$ pour $n = 3$, et combien d'évaluations faut-il au mieux (fiche §1) ?

In [ ]:
x_19, y_19 = sympy.symbols("x y")
G_EXPR_19 = x_19 ** 2 / (1 + y_19 ** 2) + sympy.cos(x_19 * y_19)      # g, for SymPy


def g_19(v):
    """g(x, y) = x² / (1 + y²) + cos(x y), for v = (x, y) (NumPy)."""
    x, y = v
    return x ** 2 / (1 + y ** 2) + np.cos(x * y)


def s_19(v):
    """s(x, y, z) = cosh x + cos y + x z + z², for v = (x, y, z): the origin is a critical point."""
    x, y, z = v
    return np.cosh(x) + np.cos(y) + x * z + z ** 2


P_19 = np.array([1.2, -0.5])                     # the point of a) and e)
STEPS_19 = [10.0 ** -k for k in range(2, 9)]     # h = 1e-2, 1e-3, ..., 1e-8

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M19")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M19, "numerical_hessian or module_uses")

**L'usage** : la hessienne d'une fonction en un point, un point critique de $\mathbb{R}^3$, et le choix du pas.

In [ ]:
answer_M19_19a = answer_M19_19b = answer_M19_19c = answer_M19_19d = answer_M19_19e = ...
with wbm.attempt("M19.19"):          # if you call numerical_hessian here: ⏳ while it is not written
    answer_M19_19a = ...  # a) a matrix (4 decimals): the numerical Hessian of g_19 at P_19
    answer_M19_19b = ...  # b) a list of 3 numbers (4 decimals), increasing: the eigenvalues of the Hessian of s_19 at 0
    answer_M19_19c = ...  # c) an integer: the index of this critical point (its negative eigenvalues)
    answer_M19_19d = ...  # d) "A", "B", "C" or "D": its nature
    answer_M19_19e = ...  # e) an integer k: the best step h = 10^(-k) among STEPS_19, for g_19 at P_19

In [ ]:
for letter, answer in zip("ab", [answer_M19_19a, answer_M19_19b]):
    wbm.check(f"M19.19{letter}", answer, computed=True)
for letter, answer in zip("cde", [answer_M19_19c, answer_M19_19d, answer_M19_19e]):
    wbm.check(f"M19.19{letter}", answer)

### Ex M19.20 — Le pas critique d'une matrice symétrique définie positive, et les itérés de Richardson · `critical_step_size, richardson_iterates` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer le pas critique $2/\lambda_{\max}$ et l'itération de Richardson, puis lire sur les itérés le facteur de convergence et le nombre d'itérations.  
**Prérequis :** fiche §4 et §5 · **mymath :** `iterative.py` · **Parcours :** C, E

Écris `critical_step_size(A)` et `richardson_iterates(A, b, omega, x0=None, n_steps=10)` (fiche §4), en suivant leurs docstrings :
- `critical_step_size(A)` rend le pas critique $\frac{2}{\lambda_{\max}}$ d'une matrice symétrique définie positive, en `float` ; `ValueError` si `A` n'est pas carrée, pas symétrique (à la tolérance de la docstring du module), ou pas définie positive ;
- `richardson_iterates(A, b, omega, x0=None, n_steps=10)` rend le tableau $(N + 1, n)$ des itérés $\mathbf{x}_0, \ldots, \mathbf{x}_N$ de $\mathbf{x}_{k+1} = \mathbf{x}_k + \omega(\mathbf{b} - \mathbf{A}\mathbf{x}_k)$, un par ligne (la ligne 0 est `x0`, le vecteur nul par défaut) ; elle ne vérifie pas la convergence : avec un pas trop grand, elle rend les itérés qui explosent ; `ValueError` pour des formes qui ne vont pas ensemble ou un nombre de pas qui n'est pas un entier $\ge 0$.

**L'usage : les ressorts de la fiche.** La cellule fournie donne la matrice $\mathbf{K}$ du ressort raide et du ressort souple (§5, `K_20`), et le système des deux ressorts (§4) : $\mathbf{A}$, $\mathbf{b}$ et sa solution $\mathbf{x}^\star = (1 ;\ 1{,}5)$ (`A_20`, `B_20`, `X_STAR_20`). Avec tes fonctions, calcule :
- a) le pas critique de $\mathbf{K}$ (6 décimales) ;
- b) le pas critique de $\tilde{\mathbf{K}} = \mathbf{D}^{-1/2}\mathbf{K}\mathbf{D}^{-1/2}$, $\mathbf{D}$ la diagonale de $\mathbf{K}$ (un nombre) ;
- c) pour les deux ressorts, avec $\omega = 0{,}32$ (le pas de la fiche, juste sous le pas critique), depuis $\mathbf{0}$ : le rapport $\lVert\mathbf{e}_{61}\rVert/\lVert\mathbf{e}_{60}\rVert$ des normes de deux écarts successifs, $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}^\star$ (4 décimales) ;
- d) pour les deux ressorts, au meilleur pas $\omega^\star = \frac27$, depuis $\mathbf{0}$ : le plus petit $k$ tel que $\lVert\mathbf{e}_k\rVert \le 10^{-6}\lVert\mathbf{e}_0\rVert$ (un entier).

Dans ta copie : combien de fois la mise à l'échelle agrandit-elle le pas critique, et pourquoi ? Compare c) à $\rho(0{,}32) = \max(\lvert 1 - 0{,}32\lambda_{\min}\rvert, \lvert 1 - 0{,}32\lambda_{\max}\rvert)$ : laquelle des deux valeurs propres décide ? Calcule aussi $\lVert\mathbf{e}_3\rVert/\lVert\mathbf{e}_2\rVert$ : de quel facteur est-il proche, et pourquoi ce n'est pas le même (compare les deux composantes de $\mathbf{e}_0 = -\mathbf{x}^\star$ sur les vecteurs propres $(1, 2)$ et $(2, -1)$) ? Compare d) à la fiche (§5). Avec $\omega^\star$, pourquoi le rapport $\lVert\mathbf{e}_{k+1}\rVert/\lVert\mathbf{e}_k\rVert$ est-il le même à chaque pas, quel que soit le départ ?

In [ ]:
K_20 = np.array([[100.0, -4.0],
                 [-4.0, 4.0]])                  # the stiff and the soft spring of the fiche (§5)
A_20 = np.array([[5.0, -2.0],
                 [-2.0, 2.0]])                  # the two springs of the fiche (§4)
B_20 = np.array([2.0, 1.0])
X_STAR_20 = np.array([1.0, 1.5])               # the solution of A_20 x = B_20

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M19")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M19, "critical_step_size or richardson_iterates")

**L'usage** : les pas critiques de la fiche, puis le facteur de convergence et le nombre d'itérations, lus sur les itérés.

In [ ]:
answer_M19_20a = answer_M19_20b = answer_M19_20c = answer_M19_20d = ...
with wbm.attempt("M19.20"):          # if you call your functions here: ⏳ while they are not written
    answer_M19_20a = ...  # a) a number (6 decimals): the critical step of K_20
    answer_M19_20b = ...  # b) a number: the critical step of K̃ = D^(-1/2) K_20 D^(-1/2)
    answer_M19_20c = ...  # c) a number (4 decimals): ‖e_61‖ / ‖e_60‖ for A_20, B_20, ω = 0.32, from 0
    answer_M19_20d = ...  # d) an integer: the smallest k with ‖e_k‖ ≤ 1e-6 ‖e_0‖ at ω* = 2/7, from 0

In [ ]:
for letter, answer in zip("abcd", [answer_M19_20a, answer_M19_20b, answer_M19_20c, answer_M19_20d]):
    wbm.check(f"M19.20{letter}", answer, computed=True)

### Ex M19.21 — Une récurrence linéaire d'ordre d · `linear_recurrence` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer une récurrence linéaire d'ordre quelconque, retrouver la forme $\lvert r\rvert^k\cos k\theta$ d'une récurrence à racines complexes, et suivre une récurrence d'ordre 3 vers sa limite.  
**Prérequis :** fiche §6 ; R5 · **mymath :** `iterative.py` · **Parcours :** C

Écris `linear_recurrence(coefficients, initial, n_terms)` (fiche §6), en suivant sa docstring : les `n_terms` premiers termes de $u_{k+d} = c_1u_{k+d-1} + \cdots + c_du_k$, où `coefficients` $= [c_1, \ldots, c_d]$ ($c_1$ multiplie le terme le plus **récent**) et `initial` $= [u_0, \ldots, u_{d-1}]$, en tableau NumPy de flottants ; une récurrence $u_{k+2} = p\,u_{k+1} - q\,u_k$ s'écrit donc `coefficients = [p, -q]`. Avec moins de `n_terms` que $d$, ce sont les premières valeurs de départ ; `ValueError` si `coefficients` est vide, si `initial` n'a pas $d$ valeurs, ou si `n_terms` n'est pas un entier $\ge 0$ ; le résultat est un **nouveau** tableau.

**L'usage.** Avec ta fonction :
- a) l'exemple de la fiche (§6), $u_{k+2} = 0{,}8\,u_{k+1} - 0{,}64\,u_k$ depuis $u_0 = 1$, $u_1 = 0{,}4$ : $u_9$ (6 décimales) ;
- b) vrai ou faux : pour $0 \le k \le 39$, ses termes s'écartent de $0{,}8^k\cos\frac{k\pi}{3}$ de moins de $10^{-12}$ ;
- c) la récurrence d'ordre 3 $u_{k+3} = 0{,}5\,u_{k+2} + 0{,}3\,u_{k+1} + 0{,}2\,u_k$, depuis $u_0 = 0$, $u_1 = 10$, $u_2 = 4$ : $u_{11}$ (6 décimales).

Dans ta copie : pourquoi $u_9$ vaut-il $-0{,}8^9$ ? Pour c), les coefficients ont pour somme 1 : chaque terme est une moyenne pondérée des trois précédents. Vérifie que $1$ est racine de $r^3 - 0{,}5r^2 - 0{,}3r - 0{,}2$, factorise, et montre que les deux autres racines sont complexes, de module $\sqrt{0{,}2}$. Montre que $u_{k+2} + 0{,}5\,u_{k+1} + 0{,}2\,u_k$ ne change pas d'un rang au suivant, et déduis-en la limite exacte $\ell$ de $u_k$. De combien $u_{11}$ s'en écarte-t-il, et de combien l'écart est-il multiplié, à peu près, à chaque rang ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M19")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M19, "linear_recurrence")

**L'usage** : une oscillation amortie de la fiche, puis une récurrence d'ordre 3.

In [ ]:
answer_M19_21a = answer_M19_21b = answer_M19_21c = ...
with wbm.attempt("M19.21"):          # if you call linear_recurrence here: ⏳ while it is not written
    answer_M19_21a = ...  # a) a number (6 decimals): u_9 of u_{k+2} = 0.8 u_{k+1} − 0.64 u_k from (1, 0.4)
    answer_M19_21b = ...  # b) True or False: |u_k − 0.8^k cos(kπ/3)| < 1e-12 for 0 ≤ k ≤ 39
    answer_M19_21c = ...  # c) a number (6 decimals): u_11 of the order-3 recurrence from (0, 10, 4)

In [ ]:
for letter, answer in zip("abc", [answer_M19_21a, answer_M19_21b, answer_M19_21c]):
    wbm.check(f"M19.21{letter}", answer, computed=True)

## Partie B · Comparer, prédire, mesurer

Fiche §3 à §5. Le gradient conjugué de SciPy contre Richardson sur une barre ; le nombre d'itérations de Richardson selon le pas, prédit puis compté ; et sa croissance avec le conditionnement.

### Ex M19.22 — `scipy.sparse.linalg.cg` : le gradient conjugué contre Richardson 📦 ★ ⏱️ 15 min
**Objectif :** résoudre un système creux avec le gradient conjugué de SciPy, compter ses itérations, et les comparer à celles de Richardson au meilleur pas.  
**Prérequis :** fiche §3 et §5 ; Ex M19.20 · **Parcours :** C

Le **gradient conjugué** (fiche §5, 💻) résout un système symétrique défini positif en choisissant, à chaque pas, la meilleure combinaison des directions déjà explorées : en arithmétique exacte, il trouve la solution en au plus $n$ itérations, et son nombre d'itérations croît comme $\sqrt\kappa$ au lieu de $\kappa$. SciPy l'implémente dans `scipy.sparse.linalg.cg`, qui accepte une matrice **creuse** (*sparse*), dont SciPy ne stocke que les coefficients non nuls.

**Une barre qui tiédit, plus finement.** Une barre a $n = 50$ points intérieurs régulièrement espacés, et ses bouts sont maintenus à 10 °C et 70 °C (`N_22`, `T_ENDS_22`). À l'équilibre, chaque température est la moyenne de ses deux voisines (fiche §3) : $-T_{i-1} + 2T_i - T_{i+1} = 0$, les températures des bouts passant au second membre. C'est $\mathbf{A}\mathbf{T} = \mathbf{b}$, avec $\mathbf{A}$ tridiagonale (2 sur la diagonale, $-1$ juste au-dessus et juste au-dessous) et $\mathbf{b} = (10, 0, \ldots, 0, 70)$. Les valeurs propres de $\mathbf{A}$ sont $\lambda_k = 2 - 2\cos\frac{k\pi}{n + 1}$, $k = 1, \ldots, n$.
- Construis `A_22` avec `scipy.sparse.diags([-np.ones(n - 1), 2 * np.ones(n), -np.ones(n - 1)], [-1, 0, 1], format="csr")`, et `B_22` ;
- lance `scipy.sparse.linalg.cg(A_22, B_22, rtol=1e-8, callback=...)` depuis $\mathbf{0}$ (le départ par défaut) : `rtol` est le résidu relatif visé, $\lVert\mathbf{b} - \mathbf{A}\mathbf{T}\rVert/\lVert\mathbf{b}\rVert \le 10^{-8}$ (l'ancien nom `tol` a disparu de SciPy), et le `callback` est une fonction que `cg` appelle une fois par itération, avec l'itéré courant : qu'elle ajoute 1 à un compteur ;
- puis lance Richardson au meilleur pas $\omega^\star = \frac{2}{\lambda_{\min} + \lambda_{\max}}$ avec ta `richardson_iterates`, 10 000 pas depuis $\mathbf{0}$ (elle attend un tableau NumPy : passe-lui `A_22.toarray()`), et cherche le premier itéré dont le résidu relatif est $\le 10^{-8}$.

Réponds :
- a) le nombre d'itérations du gradient conjugué (un entier) ;
- b) le nombre d'itérations de Richardson au meilleur pas (un entier) ;
- c) le conditionnement $\kappa$ de `A_22`, par la formule de ses valeurs propres (1 décimale).

Dans ta copie : que vaut le meilleur pas ici, et pourquoi Richardson est-elle alors la méthode de Jacobi de la fiche (§5) ? Compare a) à $n$ et à $\sqrt\kappa$, b) à $1{,}15\,\kappa\,d$ avec $d = 8$. Combien de coefficients la matrice creuse stocke-t-elle (`A_22.nnz`), contre $n^2$ pour un tableau plein ? Vérifie que la solution de `cg` est le profil linéaire entre 10 °C et 70 °C.

In [ ]:
N_22 = 50                                        # the inner points of the bar
T_ENDS_22 = (10.0, 70.0)                         # the temperatures of its two ends, in °C

In [ ]:
A_22 = ...            # the sparse (50, 50) matrix tridiag(−1, 2, −1), by scipy.sparse.diags
B_22 = ...            # the right-hand side: 10 first, 70 last, 0 elsewhere
answer_M19_22a = ...  # a) an integer: the iterations of scipy.sparse.linalg.cg down to a relative residual of 1e-8
answer_M19_22c = ...  # c) a number (1 decimal): the condition number of A_22, from the formula of its eigenvalues
answer_M19_22b = ...
with wbm.attempt("M19.22"):          # richardson_iterates: ⏳ while it is not written
    answer_M19_22b = ...  # b) an integer: the iterations of Richardson at the best step, down to 1e-8

In [ ]:
wbm.check("M19.22a", answer_M19_22a)
wbm.check("M19.22b", answer_M19_22b, computed=True)
wbm.check("M19.22c", answer_M19_22c)

### Ex M19.23 — Prédire le nombre d'itérations de Richardson selon ω, puis vérifier 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir, par le facteur $\rho(\omega)$, quel pas fait diverger Richardson et combien d'itérations demandent les autres, puis le compter.  
**Prérequis :** fiche §3 à §5 ; Ex M19.20 · **Parcours :** C, E

La cellule fournie construit une matrice symétrique $4 \times 4$, `A_23`, de valeurs propres $2$, $5$, $11$ et $24$, un second membre `B_23` de solution `X_STAR_23`, et un départ `X0_23` dont l'écart a la même composante sur les quatre vecteurs propres. On lance Richardson avec quatre pas (`OMEGAS_23`) : (A) $\omega = \frac{1}{30}$ ; (B) $\omega = \frac{1}{13}$ ; (C) $\omega = 0{,}0875$ ; (D) $\omega = 0{,}0825$.

Prédis, **avant d'exécuter quoi que ce soit** :
- a) lequel des quatre pas ne converge pas (une lettre entre guillemets) ;
- b), c) et d) pour les pas (A), (B) et (D) : le nombre d'itérations qui gagne 6 décimales, $\lVert\mathbf{e}_k\rVert \le 10^{-6}\,\lVert\mathbf{e}_0\rVert$, tel que le donne la formule $\left\lceil\frac{6\ln 10}{-\ln\rho(\omega)}\right\rceil$ (fiche §3), avec $\rho(\omega) = \max(\lvert 1 - 2\omega\rvert, \lvert 1 - 24\omega\rvert)$ (fiche §4) : trois entiers.

L'expérience lance ta `richardson_iterates` (1 500 pas au plus), compte les itérations depuis `X0_23`, et trace $\lVert\mathbf{e}_k\rVert/\lVert\mathbf{e}_0\rVert$.

Écris tes quatre prédictions, vérifie-les, puis lance l'expérience.

In [ ]:
rng_23 = np.random.default_rng(5232)
Q_23, _ = np.linalg.qr(rng_23.normal(size=(4, 4)))            # an orthonormal basis of eigenvectors (the columns)
A_23 = Q_23 @ np.diag([2.0, 5.0, 11.0, 24.0]) @ Q_23.T         # symmetric, of eigenvalues 2, 5, 11 and 24
A_23 = (A_23 + A_23.T) / 2
X_STAR_23 = rng_23.normal(size=4)                              # the solution
B_23 = A_23 @ X_STAR_23
X0_23 = X_STAR_23 + Q_23 @ np.ones(4)                          # the start: the same component on each eigenvector
OMEGAS_23 = {"A": 1 / 30, "B": 1 / 13, "C": 0.0875, "D": 0.0825}   # the four steps

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M19_23a = ...  # a) "A", "B", "C" or "D": the step that does not converge, written BEFORE the experiment
prediction_M19_23b = ...  # b) an integer: the iterations given by the formula for (A)
prediction_M19_23c = ...  # c) an integer: the iterations given by the formula for (B)
prediction_M19_23d = ...  # d) an integer: the iterations given by the formula for (D)

In [ ]:
for letter, prediction in zip("abcd", [prediction_M19_23a, prediction_M19_23b, prediction_M19_23c,
                                         prediction_M19_23d]):
    wbm.check(f"M19.23{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M19_23a, prediction_M19_23b, prediction_M19_23c, prediction_M19_23d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    with checked("M19.23"):
        print("pas    ω         ρ(ω)     ⌈6 ln 10 / (−ln ρ)⌉   itérations comptées (1 500 pas au plus)")
        fig_23, ax_23 = plt.subplots(figsize=(7.5, 4.2))
        try:
            for name_23, omega_23 in OMEGAS_23.items():
                iterates_23 = mymath.iterative.richardson_iterates(A_23, B_23, omega_23, x0=X0_23, n_steps=1500)
                gains_23 = np.linalg.norm(iterates_23 - X_STAR_23, axis=1) / np.linalg.norm(X0_23 - X_STAR_23)
                reached_23 = np.flatnonzero(gains_23 <= 1e-6)
                rho_23 = max(abs(1 - omega_23 * 2), abs(1 - omega_23 * 24))
                formula_23 = f"{math.ceil(6 * math.log(10) / -math.log(rho_23)):8d}" if rho_23 < 1 else "     ∞  "
                shown_23 = (f"{reached_23[0]}" if reached_23.size else
                            f"aucune : l'écart a été multiplié par {gains_23[-1]:.1e}")
                print(f"({name_23})    {omega_23:.5f}   {rho_23:.4f}   {formula_23}              {shown_23}")
                ax_23.semilogy(np.minimum(gains_23, 1e8), label=f"({name_23}) ω = {omega_23:.4f} : ρ = {rho_23:.3f}")
            ax_23.axhline(1e-6, color="0.5", ls="--", lw=1)
            ax_23.set_xlabel("itération k")
            ax_23.set_ylabel("‖e_k‖ / ‖e_0‖ (échelle log)")
            ax_23.set_ylim(1e-9, 1e8)
            ax_23.legend(fontsize=8)
            ax_23.grid(True, which="both", alpha=0.3)
        except BaseException:
            plt.close(fig_23)
            raise
        plt.show()

**Après l'expérience**, dans ta copie : calcule le pas critique et le meilleur pas de `A_23`, et place les quatre pas par rapport à eux. Pour chacun des trois pas qui convergent, laquelle des deux valeurs propres extrêmes décide de $\rho(\omega)$ ? Pourquoi les comptes de l'expérience sont-ils un peu en dessous de la formule (compare $\lVert\mathbf{e}_0\rVert$ à sa composante sur le vecteur propre le plus lent) ? Calcule le rapport des nombres d'itérations de (D) et de (B), alors que $\omega_D > \omega_B$.

### Ex M19.24 — Le nombre d'itérations contre le conditionnement : une droite en échelle logarithmique 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer comment le nombre d'itérations de Richardson et du gradient conjugué croît avec le conditionnement, et lire l'exposant sur une droite en échelles logarithmiques.  
**Prérequis :** fiche §5 ; Ex M19.20 et M19.22 · **Parcours :** C

Les barres de M19.22, de plus en plus fines : pour chaque nombre $n$ de points intérieurs de `N_24` (de 2 à 50), `bar_24(n)` rend la matrice tridiagonale et le second membre (bouts à 10 °C et 70 °C ; cellule fournie). Plus la barre est fine, plus le système est mal conditionné. Pour chaque barre :
- son conditionnement $\kappa$, par la formule des valeurs propres de M19.22 ;
- le nombre d'itérations de Richardson au meilleur pas, depuis $\mathbf{0}$, jusqu'au premier itéré dont le résidu relatif $\lVert\mathbf{b} - \mathbf{A}\mathbf{x}_k\rVert/\lVert\mathbf{b}\rVert$ est $\le 10^{-6}$ (ta `richardson_iterates` ; 6 000 pas suffisent) ;
- le nombre d'itérations de `scipy.sparse.linalg.cg`, avec `rtol=1e-6` (il accepte aussi un tableau plein) et un `callback` qui compte.

Range-les dans `kappas_24`, `richardson_24` et `cg_24`, dans l'ordre de `N_24`. La cellule de vérification trace les deux comptes en fonction de $\kappa$, les deux axes en échelle logarithmique, avec les droites ajustées par `np.polyfit` sur $(\ln\kappa, \ln\text{compte})$ : une droite de pente $s$ veut dire un compte proportionnel à $\kappa^s$.

Réponds :
- a) le nombre d'itérations de Richardson pour la barre la plus fine, $n = 50$ (un entier) ;
- b) la pente de la droite de Richardson, arrondie au demi le plus proche (0,5 ; 1 ; 1,5 ou 2) ;
- c) la même pente pour le gradient conjugué, arrondie au demi le plus proche.

Dans ta copie : que vaut le meilleur pas pour toutes ces barres, et pourquoi ? Compare a) à $1{,}15\,\kappa\,d$ avec $d = 6$. Pourquoi le gradient conjugué fait-il ici exactement $n$ itérations, et comment $\kappa$ grandit-il avec $n$ (pour $n$ grand, $1 - \cos x \approx \frac{x^2}{2}$) ? Pour une barre de 1 000 points, combien d'itérations prévois-tu pour chaque méthode ?

In [ ]:
N_24 = [2, 3, 5, 8, 13, 20, 32, 50]               # the inner points of eight bars, finer and finer


def bar_24(n):
    """The matrix tridiag(−1, 2, −1) of a bar with n inner points (a NumPy array), and its right-hand side, the ends
    being at 10 °C and 70 °C."""
    A = 2 * np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)
    b = np.zeros(n)
    b[0], b[-1] = 10.0, 70.0
    return A, b

In [ ]:
kappas_24 = ...       # the 8 condition numbers of the bars of N_24, from the formula of their eigenvalues
cg_24 = ...           # the 8 iteration counts of scipy.sparse.linalg.cg, down to a relative residual of 1e-6
richardson_24 = ...
answer_M19_24a = answer_M19_24b = answer_M19_24c = ...
with wbm.attempt("M19.24"):          # richardson_iterates: ⏳ while it is not written
    richardson_24 = ...   # the 8 iteration counts of Richardson at the best step, down to a relative residual of 1e-6
    answer_M19_24a = ...  # a) an integer: Richardson's count for the finest bar, n = 50
    answer_M19_24b = ...  # b) 0.5, 1, 1.5 or 2: the slope of Richardson's line, rounded to the nearest half
    answer_M19_24c = ...  # c) 0.5, 1, 1.5 or 2: the slope of the conjugate gradient's line

In [ ]:
with checked("M19.24 figure"):
    if filled(kappas_24, richardson_24, cg_24):
        fig, ax = plt.subplots(figsize=(7.0, 4.4))
        try:
            kappas_f24 = np.asarray(kappas_24, dtype=float)
            for counts_f24, label_f24, color_f24 in ((richardson_24, "Richardson au meilleur pas", "C0"),
                                                     (cg_24, "gradient conjugué", "C3")):
                counts_f24 = np.asarray(counts_f24, dtype=float)
                slope_f24, intercept_f24 = np.polyfit(np.log(kappas_f24), np.log(counts_f24), 1)
                ax.loglog(kappas_f24, counts_f24, "o", color=color_f24, label=f"{label_f24} : pente {slope_f24:.2f}")
                ax.loglog(kappas_f24, np.exp(intercept_f24) * kappas_f24 ** slope_f24, "-", color=color_f24, alpha=0.6)
            ax.set_xlabel("conditionnement κ (échelle log)")
            ax.set_ylabel("itérations jusqu'au résidu relatif 1e-6 (échelle log)")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend(fontsize=8)
        except BaseException:
            plt.close(fig)
            raise
        plt.show()
    else:
        print("⏳ Ex M19.24 : la figure attend tes trois listes, kappas_24, richardson_24 et cg_24.")
wbm.check("M19.24a", answer_M19_24a, computed=True)
for letter, answer in zip("bc", [answer_M19_24b, answer_M19_24c]):
    wbm.check(f"M19.24{letter}", answer)

**Après la figure**, dans ta copie : réponds aux questions de l'énoncé. Pour quelle taille de barre le gradient conjugué fait-il cent fois moins d'itérations que Richardson ?

## Partie C · Dessiner, déboguer, construire, défier

Fiche §1, §2, §4 et §6. Les itérés de Richardson sur les lignes de niveau ; trois erreurs classiques du chapitre ; un critère d'arrêt ; un défi : la stabilité du schéma saute-mouton d'un oscillateur. La vérification de M19.25 s'appuie sur la référence (`ref_iterative`), pour ne pas dépendre de ton code.

In [ ]:
ref_iterative = wbm.load_mymath("ref", alias="mymath_ref").iterative   # the reference: the check of M19.25

### Ex M19.25 — Les itérés de Richardson sur les lignes de niveau d'une forme quadratique 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner les itérés de Richardson sur les lignes de niveau de $\frac12\mathbf{x}^\top\mathbf{A}\mathbf{x} - \mathbf{b}^\top\mathbf{x}$, et voir l'effet du pas et du conditionnement.  
**Prérequis :** fiche §4 et §5 ; Ex M19.20 · **Parcours :** C, E

Richardson est une descente sur $f(\mathbf{x}) = \frac12\mathbf{x}^\top\mathbf{A}\mathbf{x} - \mathbf{b}^\top\mathbf{x}$ (fiche §4) : on la voit sur les lignes de niveau de $f$, des ellipses centrées en $\mathbf{x}^\star$, allongées dans la direction propre de $\lambda_{\min}$ (fiche §5). La cellule fournie donne $\mathbf{A} = \begin{pmatrix} 10 & 6 \\ 6 & 5 \end{pmatrix}$, de valeurs propres 14 et 1 ($\kappa = 14$ ; vecteurs propres $(3, 2)$ et $(2, -3)$), la solution $\mathbf{x}^\star = (1 ;\ -1)$, $\mathbf{b} = \mathbf{A}\mathbf{x}^\star$, le départ $\mathbf{x}_0 = (-2 ;\ 1)$, $N = 25$ pas, et trois pas : $0{,}4\,\omega_c$, le meilleur pas et $0{,}95\,\omega_c$ (`OMEGAS_25`).

Écris `draw_richardson(ax, A, b, omega, x0, n_steps)`, qui dessine dans l'axe `ax` (sans en ouvrir d'autre) :
- les lignes de niveau de $f$ : `ax.contour(X, Y, F, levels=...)`, avec `X, Y = np.meshgrid(xs, ys)` sur une grille qui contient tous les itérés, et `F` la valeur de $f$ en chaque point de la grille (`F = 0.5 * (A[0, 0] * X**2 + 2 * A[0, 1] * X * Y + A[1, 1] * Y**2) - b[0] * X - b[1] * Y`) ;
- les itérés $\mathbf{x}_0, \ldots, \mathbf{x}_N$ que rend ta `richardson_iterates`, reliés dans l'ordre par **une seule** ligne avec des points : `ax.plot(xs[:, 0], xs[:, 1], "o-")` ;
- la solution $\mathbf{x}^\star$, marquée d'un point (`np.linalg.solve` est permis ici) ;
- la même échelle sur les deux axes (`ax.set_aspect("equal")`), et un titre qui donne $\omega$.

La vérification appelle ta fonction pour les trois pas, côte à côte, et regarde le dessin : les lignes de niveau, une ligne qui passe par les $N + 1$ itérés de la référence, dans l'ordre, la solution marquée, et les échelles égales.

In [ ]:
A_25 = np.array([[10.0, 6.0],
                 [6.0, 5.0]])                   # symmetric positive definite, eigenvalues 14 and 1
X_STAR_25 = np.array([1.0, -1.0])               # the solution
B_25 = A_25 @ X_STAR_25
X0_25 = np.array([-2.0, 1.0])                   # the start
N_25 = 25                                       # the number of steps
OMEGA_C_25 = 2 / 14                             # the critical step
OMEGAS_25 = [0.4 * OMEGA_C_25, 2 / (1 + 14), 0.95 * OMEGA_C_25]   # 0.4 of it, the best step, just below it

In [ ]:
def draw_richardson(ax, A, b, omega, x0, n_steps):
    """In ax (and no other axes): the level lines of f(x) = ½ xᵀ A x − bᵀ x, the iterates x_0, ..., x_N of
    Richardson's method joined in order by one line with markers, the solution marked, equal scales, a title."""
    raise NotImplementedError("draw_richardson n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M19.25"):
    results_25 = []

    def report_25(ok, success, failure):
        results_25.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    fig_25, axes_25 = plt.subplots(1, 3, figsize=(14.0, 4.6))
    try:
        opened_25, contour_25, path_25, star_25, aspect_25 = [], [], [], [], []
        for ax_25, omega_25 in zip(axes_25, OMEGAS_25):
            opened_25 += drawn(draw_richardson, ax_25, A_25, B_25, omega_25, X0_25, N_25)
            want_25 = ref_iterative.richardson_iterates(A_25, B_25, omega_25, x0=X0_25, n_steps=N_25)
            contour_25.append(any(isinstance(item, ContourSet) for item in ax_25.collections))
            path_25.append(any(np.shape(line.get_xydata()) == want_25.shape
                               and np.allclose(line.get_xydata(), want_25, rtol=1e-9, atol=1e-9)
                               for line in ax_25.lines))
            points_25 = [np.asarray(line.get_xydata(), dtype=float).reshape(-1, 2) for line in ax_25.lines]
            points_25 += [np.asarray(item.get_offsets(), dtype=float).reshape(-1, 2) for item in ax_25.collections
                          if not isinstance(item, ContourSet)]
            star_25.append(any(len(p) and np.min(np.linalg.norm(p - X_STAR_25, axis=1)) < 1e-6 for p in points_25))
            aspect_25.append(ax_25.get_aspect() in ("equal", 1.0))
        report_25(not opened_25, "draw_richardson dessine dans l'axe reçu",
                  "draw_richardson doit dessiner dans l'axe reçu, sans ouvrir de figure (ni plt.figure ni plt.subplots)")
        report_25(all(contour_25), "les lignes de niveau sont tracées (ax.contour)",
                  f"pas de lignes de niveau (ax.contour) dans {contour_25.count(False)} des trois axes")
        report_25(all(path_25), f"les {N_25 + 1} itérés x₀, …, x_N sont reliés par une ligne, dans l'ordre",
                  "aucune ligne ne passe par les itérés x₀, …, x_N, tous et dans l'ordre, pour "
                  f"{path_25.count(False)} des trois pas : ax.plot(xs[:, 0], xs[:, 1], \"o-\"), avec xs le tableau "
                  "de richardson_iterates (x0, n_steps)")
        report_25(all(star_25), "la solution x* est marquée",
                  "marque la solution x* d'un point (ax.plot(x_star[0], x_star[1], \"k*\")), dans chaque axe")
        report_25(all(aspect_25), "les deux axes ont la même échelle : les ellipses ne sont pas déformées",
                  "donne la même échelle aux deux axes : ax.set_aspect(\"equal\")")
    except BaseException:
        plt.close(fig_25)
        raise
    plt.show()
    verdict("M19.25", all(results_25), "les lignes de niveau, les itérés et la solution sont dessinés pour les trois "
                                       "pas.", "relis les lignes ❌ ci-dessus.")

**Regarde ta figure**, puis, dans ta copie : pour chaque pas, que vaut le facteur $1 - \omega\lambda$ dans la direction raide $(3, 2)$, et dans la direction plate $(2, -3)$ ? Avec $0{,}95\,\omega_c$, pourquoi les itérés passent-ils de part et d'autre de l'axe long à chaque pas ? Calcule, pour $0 < \omega < \omega_c$, le plus petit facteur possible $1 - \omega\lambda_{\min}$ dans la direction de l'axe long : montre qu'il vaut $1 - \frac{2}{\kappa}$, et donne sa valeur ici. Que verrais-tu avec $1{,}05\,\omega_c$ ?

### Ex M19.26 — Chasse à l'erreur : un pas réglé sur λmin, une hessienne non symétrique, le module d'une racine complexe 🐛 ★ ⏱️ 25 min
**Objectif :** reconnaître trois erreurs classiques du chapitre qui ne lèvent aucune erreur Python et passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §1, §2, §4 et §6 · **Parcours :** C, E

Un collègue a écrit trois morceaux de code (cellule fournie) ; chacun a l'erreur que nomme le titre, aucun ne lève d'erreur Python, et chacun a passé la vérification que le collègue a faite :
1. `critical_26(A)` doit rendre le pas critique $\frac{2}{\lambda_{\max}}$ d'une matrice symétrique définie positive. Il l'a vérifiée sur $4\mathbf{I}$ : 0,5, juste. Puis il a lancé Richardson sur le système des trois masses `K_26`, `F_26` avec la moitié de ce pas, « par prudence » : les itérés explosent.
2. `hessian_26(f, x, h)` calcule la hessienne par différences centrées, chaque case de la diagonale et au-dessus une seule fois ; `inertia_26(H)` rend l'**inertie** $(n_-, n_0, n_+)$ d'une matrice symétrique, ses nombres de valeurs propres négatives, nulles et positives (`np.linalg.eigvalsh`), une valeur propre comptant pour nulle quand son module est au plus $10^{-8}$ fois le plus grand. Il les a vérifiées sur $\cos x + y^2 + z^2$ en $\mathbf{0}$ : $(1, 0, 2)$, une selle, juste.
3. `converges_26(p, q)` doit dire si toutes les solutions de $u_{k+2} = p\,u_{k+1} - q\,u_k$ tendent vers 0. Il l'a vérifiée sur les trois récurrences de la fiche (§6), $(p, q) = (1 ;\ 0{,}21)$, $(0{,}8 ;\ 0{,}64)$ et $(1{,}5 ;\ -1)$ : juste.

À faire :
- a) avec le pas du collègue, $\omega$ = `critical_26(K_26) / 2` : le facteur $\rho(\omega) = \max_i\lvert 1 - \omega\lambda_i\rvert$ sur les valeurs propres de `K_26` (un nombre, 4 décimales) ;
- (1) écris `critical_26_fixed(A)` ;
- (2) écris `hessian_26_fixed(f, x, h=1e-4)` ; b) avec elle et `inertia_26`, la nature du point critique $\mathbf{0}$ de $g(x, y, z) = x^2 + y^2 + z^2 + 3\sin x\sin y + yz$ (`g_26`) : (A) minimum local ; (B) maximum local ; (C) point selle ; (D) le test ne conclut pas (une lettre entre guillemets) ;
- (3) écris `converges_26_fixed(p, q)` ; c) le module des racines de $r^2 - r + 1{,}2$, c'est-à-dire pour $(p, q) = (1 ;\ 1{,}2)$ (4 décimales).

La vérification essaie tes corrections sur des cas que tu ne vois pas : d'autres matrices, d'autres fonctions et d'autres points, une grille de couples $(p, q)$.

In [ ]:
K_26 = np.array([[3.0, -2.0, 0.0],
                 [-2.0, 5.0, -3.0],
                 [0.0, -3.0, 9.0]])            # three masses, four springs of stiffness 1, 2, 3 and 6
F_26 = np.array([1.0, 0.0, 2.0])               # the forces on the masses


# The colleague's code
def critical_26(A):
    """The critical step 2 / λmax of a symmetric positive definite matrix A."""
    return 2.0 / np.linalg.eigvalsh(A)[0]


def hessian_26(f, x, h=1e-4):
    """The Hessian of f at x by centered differences: each entry on or above the diagonal is computed once."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    E = h * np.eye(n)
    H = np.zeros((n, n))
    for i in range(n):
        H[i, i] = (f(x + E[i]) - 2 * f(x) + f(x - E[i])) / h ** 2
        for j in range(i + 1, n):
            H[i, j] = (f(x + E[i] + E[j]) - f(x + E[i] - E[j]) - f(x - E[i] + E[j]) + f(x - E[i] - E[j])) / (4 * h ** 2)
    return H


def inertia_26(H):
    """The inertia (n_neg, n_zero, n_pos) of a symmetric matrix H: its numbers of negative, zero and positive
    eigenvalues, an eigenvalue counting as zero when its modulus is at most 1e-8 times the largest modulus."""
    values = np.linalg.eigvalsh(H)
    zero = 1e-8 * np.max(np.abs(values))
    return int(np.sum(values < -zero)), int(np.sum(np.abs(values) <= zero)), int(np.sum(values > zero))


def converges_26(p, q):
    """True when every solution of u_{k+2} = p u_{k+1} − q u_k tends to 0: both roots of r² − p r + q have a modulus
    < 1."""
    delta = p * p - 4 * q
    if delta >= 0:                                     # two real roots
        roots = [(p + math.sqrt(delta)) / 2, (p - math.sqrt(delta)) / 2]
        return max(abs(r) for r in roots) < 1
    modulus = abs(p) / 2                               # complex roots p/2 ± i √(−Δ)/2
    return modulus < 1


def g_26(v):
    """g(x, y, z) = x² + y² + z² + 3 sin x sin y + y z, for v = (x, y, z): the origin is a critical point."""
    x, y, z = v
    return x ** 2 + y ** 2 + z ** 2 + 3 * np.sin(x) * np.sin(y) + y * z


# The colleague's checks: all pass
print("(1) critical_26(4 I) =", critical_26(4 * np.eye(3)))
print("(2) inertia_26 de la hessienne de cos x + y² + z² en 0 :",
      inertia_26(hessian_26(lambda v: np.cos(v[0]) + v[1] ** 2 + v[2] ** 2, np.zeros(3))), "(une selle)")
print("(3) converges_26 pour les trois récurrences de la fiche :",
      [converges_26(p, q) for p, q in ((1.0, 0.21), (0.8, 0.64), (1.5, -1.0))])
omega_26 = critical_26(K_26) / 2                 # « half the critical step, to be safe »
x_26 = np.zeros(3)
for k_26 in range(1, 31):
    x_26 = x_26 + omega_26 * (F_26 - K_26 @ x_26)
    if k_26 % 10 == 0:
        print(f"(1) Richardson sur K_26 x = F_26 avec ce pas, itération {k_26} : ‖x‖ = {np.linalg.norm(x_26):.3g}")

In [ ]:
answer_M19_26a = ...  # a) a number (4 decimals): ρ(ω) = max |1 − ω λ_i| over the eigenvalues of K_26, for the colleague's ω


def critical_26_fixed(A):
    """The critical step 2 / λmax of a symmetric positive definite matrix A."""
    raise NotImplementedError("critical_26_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def hessian_26_fixed(f, x, h=1e-4):
    """The Hessian of f at x by centered differences, exactly symmetric."""
    raise NotImplementedError("hessian_26_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def converges_26_fixed(p, q):
    """True when every solution of u_{k+2} = p u_{k+1} − q u_k tends to 0: both roots of r² − p r + q have a modulus
    < 1."""
    raise NotImplementedError("converges_26_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M19_26c = ...  # c) a number (4 decimals): the modulus of the roots of r² − r + 1.2
answer_M19_26b = ...
with wbm.attempt("M19.26"):          # hessian_26_fixed: ⏳ while it is not written
    answer_M19_26b = ...  # b) "A", "B", "C" or "D": the nature of the critical point 0 of g_26, by your Hessian

In [ ]:
for letter, answer in zip("abc", [answer_M19_26a, answer_M19_26b, answer_M19_26c]):
    wbm.check(f"M19.26{letter}", answer)

results_26, parts_26 = [], []


def report_26(ok, success, failure):
    results_26.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


with checked("M19.26"):                                       # (1) critical_26_fixed
    rng_26 = np.random.default_rng(5236)
    wrong_26 = []
    for n_26 in (1, 2, 3, 5):
        Q_c26, _ = np.linalg.qr(rng_26.normal(size=(n_26, n_26)))
        values_c26 = np.sort(rng_26.uniform(0.3, 9.0, size=n_26))
        A_c26 = Q_c26 @ np.diag(values_c26) @ Q_c26.T
        A_c26 = (A_c26 + A_c26.T) / 2
        got_26 = critical_26_fixed(A_c26.copy())
        if got_26 is None or not math.isclose(float(got_26), 2 / values_c26[-1], rel_tol=1e-9):
            wrong_26.append(f"une matrice {n_26} × {n_26} de valeurs propres {np.round(values_c26, 2).tolist()} : "
                            f"attendu {2 / values_c26[-1]:.6g}, reçu {got_26!r}"[:200])
    report_26(not wrong_26, "(1) critical_26_fixed : le pas critique de quatre matrices",
              "(1) critical_26_fixed, " + " ; ".join(wrong_26[:2]))
    parts_26.append(1)

with checked("M19.26"):                                       # (2) hessian_26_fixed
    xyz_26 = sympy.symbols("x y z")
    cases_26 = [(sympy.sin(xyz_26[0] * xyz_26[1]) + xyz_26[0] * xyz_26[2] ** 2, 3),
                (sympy.log(2 + xyz_26[0] ** 2) + xyz_26[1] * xyz_26[2] ** 3, 3),
                (xyz_26[0] ** 3 * xyz_26[1] - sympy.cos(xyz_26[1]), 2),
                (xyz_26[0] ** 2 + xyz_26[1] ** 2 + xyz_26[2] ** 2 + 3 * sympy.sin(xyz_26[0]) * sympy.sin(xyz_26[1])
                 + xyz_26[1] * xyz_26[2], 3)]                 # g_26
    rng_26 = np.random.default_rng(5237)
    wrong_26 = []
    for expr_26, n_26 in cases_26:
        symbols_26 = xyz_26[:n_26]
        f_c26 = sympy.lambdify([symbols_26], expr_26, "numpy")
        point_26 = rng_26.uniform(-1.5, 1.5, size=n_26)
        exact_26 = np.array(sympy.hessian(expr_26, symbols_26).subs(dict(zip(symbols_26, point_26))), dtype=float)
        got_26 = hessian_26_fixed(f_c26, point_26.copy())
        shape_26 = np.shape(got_26) if got_26 is not None else None
        if shape_26 != (n_26, n_26):
            wrong_26.append(f"pour {expr_26} : attendu une matrice {n_26} × {n_26}, reçu {got_26!r}"[:160])
        elif not np.array_equal(np.asarray(got_26), np.asarray(got_26).T):
            wrong_26.append(f"pour {expr_26} : la matrice n'est pas exactement symétrique")
        elif not np.allclose(got_26, exact_26, rtol=1e-5, atol=1e-5):
            wrong_26.append(f"pour {expr_26} en {np.round(point_26, 3).tolist()} : attendu "
                            f"{np.round(exact_26, 4).tolist()}, reçu {np.round(got_26, 4).tolist()}"[:240])
    report_26(not wrong_26, "(2) hessian_26_fixed : quatre fonctions en des points tirés au hasard, symétrique",
              "(2) hessian_26_fixed, " + " ; ".join(wrong_26[:2]))
    exact_g26 = np.array(sympy.hessian(cases_26[-1][0], xyz_26).subs(dict.fromkeys(xyz_26, 0)), dtype=float)
    seen_26 = inertia_26(hessian_26_fixed(g_26, np.zeros(3)))
    report_26(seen_26 == inertia_26(exact_g26), "(2) avec hessian_26_fixed, inertia_26 lit en 0 la même inertie que sur "
                                               "la hessienne exacte de g_26",
              f"(2) avec hessian_26_fixed, inertia_26 rend {seen_26!r} pour g_26 en 0, autre chose qu'avec la hessienne "
              "exacte")
    parts_26.append(2)

with checked("M19.26"):                                       # (3) converges_26_fixed
    wrong_26, tried_26 = [], 0
    for p_26 in np.linspace(-2.5, 2.5, 21):
        for q_26 in np.linspace(-1.35, 1.65, 21):
            if min(abs(abs(q_26) - 1), abs(abs(p_26) - (1 + q_26))) < 1e-9:
                continue                                      # on the border of the triangle: not tried
            tried_26 += 1
            want_26 = bool(np.max(np.abs(np.roots([1.0, -p_26, q_26]))) < 1)
            got_26 = converges_26_fixed(float(p_26), float(q_26))
            if got_26 is None or bool(got_26) != want_26:
                wrong_26.append(f"(p, q) = ({p_26:.2f} ; {q_26:.2f}) : attendu {want_26}, reçu {got_26!r}")
    report_26(not wrong_26, f"(3) converges_26_fixed : juste sur une grille de {tried_26} couples (p, q)",
              f"(3) converges_26_fixed se trompe pour {len(wrong_26)} couple(s) sur {tried_26}, par exemple "
              + " ; ".join(wrong_26[:2]))
    parts_26.append(3)

if len(parts_26) == 3:
    verdict("M19.26", all(results_26), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : sur $4\mathbf{I}$, pourquoi la vérification du collègue ne pouvait-elle pas voir l'erreur (1) ? Que fait `np.linalg.eigvalsh` d'une matrice dont seul le triangle supérieur est rempli (lis la documentation de son argument `UPLO`), et pourquoi la fonction de test du collègue, $\cos x + y^2 + z^2$, ne pouvait-elle pas voir l'erreur (2) ? Les trois récurrences de la fiche passent le test (3) : pour laquelle des trois $\lvert p\rvert/2$ et $\sqrt q$ sont-ils tous deux $< 1$, et quelles autres récurrences auraient démasqué l'erreur ?

### Ex M19.27 — Un critère d'arrêt : le résidu relatif ‖b − A x‖/‖b‖ et un nombre maximal d'itérations 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire une itération de Richardson qui suit son résidu par sa propre récurrence, s'arrête d'elle-même au premier itéré assez précis, et sait dire qu'elle n'a pas convergé.  
**Prérequis :** fiche §3 et §4 (💻) ; Ex M19.20 ; M12d · **Parcours :** C

Une méthode itérative utilisable ne fait pas un nombre de pas fixé d'avance : elle s'arrête quand le **résidu relatif** $\frac{\lVert\mathbf{b} - \mathbf{A}\mathbf{x}_k\rVert}{\lVert\mathbf{b}\rVert}$ passe sous une tolérance, et abandonne après un nombre maximal d'itérations (fiche §4, 💻). Calculer $\mathbf{b} - \mathbf{A}\mathbf{x}_k$ à chaque itération en plus du pas coûterait deux produits par $\mathbf{A}$ ; le résidu $\mathbf{r}_k = \mathbf{b} - \mathbf{A}\mathbf{x}_k$ a sa **propre récurrence** : de $\mathbf{x}_{k+1} = \mathbf{x}_k + \omega\,\mathbf{r}_k$, on tire $\mathbf{r}_{k+1} = \mathbf{r}_k - \omega\mathbf{A}\mathbf{r}_k = (\mathbf{I} - \omega\mathbf{A})\,\mathbf{r}_k$, et $\mathbf{r}_0 = \mathbf{b}$ depuis $\mathbf{x}_0 = \mathbf{0}$. Un seul produit par $\mathbf{A}$ par itération suffit.

Écris `solve_richardson(A, b, omega, tol=1e-8, max_iter=10_000)`, qui part de $\mathbf{x}_0 = \mathbf{0}$, $\mathbf{r}_0 = \mathbf{b}$, et rend le triplet `(x, n_iter, converged)` ; pour $k = 0, 1, 2, \ldots$, elle regarde le résidu relatif $\frac{\lVert\mathbf{r}_k\rVert}{\lVert\mathbf{b}\rVert}$ :
- s'il est $\le$ `tol` : elle rend `(x_k, k, True)` ;
- sinon, s'il n'est plus un nombre fini, ou s'il dépasse $10^{10}$ (l'itération diverge : inutile d'attendre l'overflow), ou si $k$ = `max_iter` : elle rend `(x_k, k, False)` ;
- sinon : $\mathbf{p} = \mathbf{A}\mathbf{r}_k$ (le seul produit), $\mathbf{x}_{k+1} = \mathbf{x}_k + \omega\,\mathbf{r}_k$ et $\mathbf{r}_{k+1} = \mathbf{r}_k - \omega\,\mathbf{p}$.

`b` n'est pas nul ; `A` et `b` ne sont pas modifiés ; aucun `RuntimeWarning` (un overflow) ne doit apparaître ; un pas qui n'est pas un nombre (`np.nan`) doit lui aussi s'arrêter, sans boucle infinie, avec `converged` faux.

La cellule fournie donne le système de quatre masses entre deux murs, reliées par cinq ressorts, `K_27` $\mathbf{x}$ = `F_27`. La vérification lance ta fonction sur ce système avec un bon pas ($\frac{1}{\lambda_{\max}}$), avec `tol=1e-3`, avec `max_iter=15`, avec un pas trop grand ($1{,}2\,\omega_c$), avec `omega=np.nan`, puis sur trois systèmes tirés au hasard ; elle compare chaque triplet à celui de la règle ci-dessus, et la solution de `K_27` à `np.linalg.solve` (permis ici, pour vérifier).

Dans ta copie :
1. Démontre $\mathbf{r}_{k+1} = (\mathbf{I} - \omega\mathbf{A})\,\mathbf{r}_k$, puis, depuis $\mathbf{x}_0 = \mathbf{0}$ et pour $\mathbf{A}$ symétrique, $\lVert\mathbf{r}_k\rVert \le \rho(\omega)^k\,\lVert\mathbf{b}\rVert$ (§3 : $\mathbf{I} - \omega\mathbf{A}$ est symétrique, de valeurs propres $1 - \omega\lambda_i$). Pourquoi un résidu relatif au-dessus de 1, a fortiori de $10^{10}$, prouve-t-il que $\rho(\omega) > 1$ ?
2. Avec cette borne, combien d'itérations au plus faut-il pour `K_27` avec $\omega = \frac{1}{\lambda_{\max}}$ et `tol=1e-8` ? Compare au compte de ta fonction.
3. Montre que $\frac{\lVert\mathbf{x}_k - \mathbf{x}^\star\rVert}{\lVert\mathbf{x}^\star\rVert} \le \kappa\,\frac{\lVert\mathbf{r}_k\rVert}{\lVert\mathbf{b}\rVert}$ (avec $\mathbf{x}_k - \mathbf{x}^\star = -\mathbf{A}^{-1}\mathbf{r}_k$ et $\lVert\mathbf{b}\rVert \le \lambda_{\max}\lVert\mathbf{x}^\star\rVert$ ; M12d). Que vaut $\kappa$ pour `K_27`, et que garantit `tol=1e-8` sur l'erreur relative de $\mathbf{x}$ ? Que deviendrait cette garantie pour un système de conditionnement $10^6$ ?

In [ ]:
K_27 = np.array([[4.0, -1.0, 0.0, 0.0],
                 [-1.0, 3.0, -2.0, 0.0],
                 [0.0, -2.0, 3.0, -1.0],
                 [0.0, 0.0, -1.0, 4.0]])       # four masses, five springs of stiffness 3, 1, 2, 1, 3 between two walls
F_27 = np.array([1.0, 0.0, 0.0, 2.0])         # the forces on the masses

In [ ]:
def solve_richardson(A, b, omega, tol=1e-8, max_iter=10_000):
    """Richardson's iteration from x_0 = 0 for A x = b, its residual r_k = b − A x_k updated by r_{k+1} = r_k − ω A r_k
    (one product by A per iteration), stopped at the first k whose relative residual ‖r_k‖ / ‖b‖ is ≤ tol (converged),
    or is not finite or above 1e10, or when k = max_iter (not converged). Returns (x_k, k, converged)."""
    raise NotImplementedError("solve_richardson n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M19.27"):
    results_27 = []

    def report_27(ok, success, failure):
        results_27.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def expected_27(A, b, omega, tol, max_iter):
        """The expected (x, n_iter, converged), by the rule of the statement (the residual by its own recurrence)."""
        x, r = np.zeros(len(b)), b.copy()
        for k in range(max_iter + 1):
            relative = np.linalg.norm(r) / np.linalg.norm(b)
            if relative <= tol:
                return x, k, True
            if not np.isfinite(relative) or relative > 1e10 or k == max_iter:
                return x, k, False
            product = A @ r
            x, r = x + omega * r, r - omega * product

    def run_27(A, b, omega, **options):
        """solve_richardson on copies of A and b, a RuntimeWarning turned into an error, the triple checked."""
        A_copy, b_copy = A.copy(), b.copy()
        with warnings.catch_warnings():
            warnings.simplefilter("error", RuntimeWarning)
            try:
                result = solve_richardson(A_copy, b_copy, omega, **options)
            except RuntimeWarning as warning:
                print(f"❌ Ex M19.27 : un RuntimeWarning ({warning}) : arrête la boucle dès que le résidu relatif "
                      "n'est plus fini ou dépasse 1e10, avant l'overflow")
                raise StopCheck from None
        if not (isinstance(result, tuple) and len(result) == 3):
            print(f"❌ Ex M19.27 : solve_richardson doit rendre le triplet (x, n_iter, converged), reçu {result!r}"[:200])
            raise StopCheck
        if not (np.array_equal(A_copy, A) and np.array_equal(b_copy, b)):
            print("❌ Ex M19.27 : solve_richardson a modifié A ou b : travaille sur des copies")
            raise StopCheck
        return np.asarray(result[0], dtype=float), result[1], result[2]

    def same_27(got, want):
        return (got[1] == want[1] and bool(got[2]) == want[2] and got[0].shape == want[0].shape
                and np.allclose(got[0], want[0], rtol=1e-9, atol=1e-12))

    def shown_27(triple):
        return f"(x, {triple[1]!r}, {triple[2]!r})"

    lambdas_27 = np.linalg.eigvalsh(K_27)
    good_27, bad_27 = 1 / lambdas_27[-1], 1.2 * 2 / lambdas_27[-1]
    cases_27 = [("un bon pas, 1/λmax", K_27, F_27, good_27, {}),
                ("tol = 1e-3", K_27, F_27, good_27, {"tol": 1e-3}),
                ("max_iter = 15", K_27, F_27, good_27, {"max_iter": 15}),
                ("un pas trop grand, 1,2 ω_c", K_27, F_27, bad_27, {})]
    rng_27 = np.random.default_rng(5238)
    for n_27 in (3, 6, 10):                                   # systems drawn at random, at their best step
        Q_27, _ = np.linalg.qr(rng_27.normal(size=(n_27, n_27)))
        values_27 = np.linspace(0.5, rng_27.uniform(4.0, 12.0), n_27)
        A_r27 = Q_27 @ np.diag(values_27) @ Q_27.T
        A_r27 = (A_r27 + A_r27.T) / 2
        cases_27.append((f"un système {n_27} × {n_27} tiré au hasard", A_r27, rng_27.normal(size=n_27),
                         2 / (values_27[0] + values_27[-1]), {}))
    for label_27, A_c27, b_c27, omega_c27, options_27 in cases_27:
        want_27 = expected_27(A_c27, b_c27, omega_c27, options_27.get("tol", 1e-8), options_27.get("max_iter", 10_000))
        got_27 = run_27(A_c27, b_c27, omega_c27, **options_27)
        report_27(same_27(got_27, want_27), f"{label_27} : {shown_27(want_27)}, avec le bon x",
                  f"{label_27} : attendu {shown_27(want_27)}, reçu {shown_27(got_27)}"
                  + ("" if got_27[1] != want_27[1] or got_27[2] != want_27[2] else " avec un autre x"))
    _, n_nan_27, converged_nan_27 = run_27(K_27, F_27, np.nan)
    report_27(not converged_nan_27 and n_nan_27 <= 1,
              f"omega = nan : arrêt après {n_nan_27} pas, converged = {converged_nan_27!r}",
              f"omega = nan : attendu un arrêt dès que le résidu n'est plus un nombre fini (au plus 1 pas) avec "
              f"converged = False, reçu {n_nan_27!r} pas et converged = {converged_nan_27!r}")
    x_27, n_27, converged_27 = run_27(K_27, F_27, good_27)
    gap_27 = np.linalg.norm(x_27 - np.linalg.solve(K_27, F_27)) / np.linalg.norm(np.linalg.solve(K_27, F_27))
    report_27(converged_27 and gap_27 < 1e-7, f"pour K_27 x = F_27, x s'écarte de np.linalg.solve de {gap_27:.1e} en "
                                              f"relatif, après {n_27} itérations",
              f"pour K_27 x = F_27, x s'écarte de np.linalg.solve de {gap_27:.1e} en relatif")
    verdict("M19.27", all(results_27), "le critère d'arrêt s'arrête au bon itéré dans tous les cas, sans warning.",
            "relis les lignes ❌ ci-dessus.")

### Ex M19.28 — Défi : le schéma saute-mouton d'un oscillateur, x_{k+1} = (2 − h²Ω²) x_k − x_{k−1}, borné si et seulement si hΩ < 2 🏆 ★★★ ⏱️ 30 min
**Objectif :** lire la stabilité d'un schéma de la mécanique sur les racines d'une récurrence d'ordre 2, puis la vérifier par la simulation, jusqu'au cas limite.  
**Prérequis :** fiche §6 ; M05a ; Ex M19.21 · **Parcours :** C

Une masse au bout d'un ressort oscille selon $\ddot x = -\Omega^2x$, de pulsation $\Omega$ (en rad/s). Le **schéma saute-mouton** (*leapfrog*), courant dans les simulations de mécanique, remplace la dérivée seconde par la différence centrée de M05a, avec un pas de temps $h$ : $\frac{x_{k+1} - 2x_k + x_{k-1}}{h^2} = -\Omega^2x_k$, soit
$$x_{k+1} = (2 - h^2\Omega^2)\,x_k - x_{k-1},$$
une récurrence d'ordre 2 (fiche §6) avec $p = 2 - h^2\Omega^2$ et $q = 1$. On lâche la masse de sa position d'équilibre avec la vitesse $v_0$ : $x_0 = 0$ et $x_1 = h\,v_0$. La cellule fournie donne $\Omega = 5$ rad/s (`OMEGA_28`), $v_0 = 1{,}5$ m/s (`V0_28`) et quinze pas de temps $h = 0{,}04 ;\ 0{,}08 ;\ \ldots ;\ 0{,}60$ s (`H_28`). L'oscillateur exact a l'amplitude $\frac{v_0}{\Omega} = 0{,}3$ m.

**L'objectif.**
- a) Par la théorie du §6 : les racines de $r^2 - p\,r + 1$ ont pour produit 1 ; elles sont complexes conjuguées, de module 1, quand $\lvert p\rvert < 2$, c'est-à-dire pour $0 < h\Omega < 2$, soit $0 < h < h_c$. Le pas critique $h_c$ (un nombre).
- b) Par la simulation, avec ta `linear_recurrence` : pour chaque $h$ de `H_28`, les 301 termes $x_0, \ldots, x_{300}$, et leur amplitude $\max_k\lvert x_k\rvert$. On dit la suite **bornée** quand cette amplitude ne dépasse pas 1 m. Le plus grand $h$ de `H_28` pour lequel elle l'est (un nombre).
- c) Pour un petit pas, $h = 0{,}03$ s : la période des oscillations du schéma, en pas, $\frac{2\pi}{\theta}$ avec $\cos\theta = 1 - \frac{h^2\Omega^2}{2}$ (2 décimales).
- d) En $h = h_c$ : $\lvert x_{1000}\rvert$, par la simulation (un nombre, 6 décimales).

Dans ta copie : pour $h\Omega > 2$, les racines sont réelles et de produit 1 : pourquoi l'une a-t-elle un module $> 1$ ? En $h\Omega = 2$, quelle est la racine double, et pourquoi $x_k = (\alpha + \beta k)(-1)^k$ grandit-il ici (que vaut $\beta$ pour ce départ) ? Compare c) à la période de l'oscillateur exact, $\frac{2\pi}{\Omega}$, comptée en pas, $\frac{2\pi}{h\Omega}$. Le schéma amortit-il l'oscillation, comme une récurrence de module $< 1$ ? Quel pas de `H_28` garderais-tu pour simuler cet oscillateur, et pourquoi pas le plus grand pas borné ?

In [ ]:
OMEGA_28 = 5.0                                   # the angular frequency of the oscillator, in rad/s
V0_28 = 1.5                                      # the speed at t = 0, in m/s, from the equilibrium x = 0
H_28 = [round(0.04 * k, 2) for k in range(1, 16)]   # the time steps h = 0.04, 0.08, ..., 0.60 s

In [ ]:
answer_M19_28a = ...  # a) a number: the critical time step h_c, by the theory of §6
answer_M19_28c = ...  # c) a number (2 decimals): the period 2π/θ, in steps, for h = 0.03
answer_M19_28b = answer_M19_28d = ...
with wbm.attempt("M19.28"):          # linear_recurrence: ⏳ while it is not written
    answer_M19_28b = ...  # b) a number: the largest h of H_28 whose 301 terms stay within 1 m
    answer_M19_28d = ...  # d) a number (6 decimals): |x_1000| for h = h_c

In [ ]:
for letter, answer in zip("ac", [answer_M19_28a, answer_M19_28c]):
    wbm.check(f"M19.28{letter}", answer)
for letter, answer in zip("bd", [answer_M19_28b, answer_M19_28d]):
    wbm.check(f"M19.28{letter}", answer, computed=True)

**Dans ta copie**, réponds aux questions de l'énoncé, puis trace l'amplitude en fonction de $h\Omega$, en échelle logarithmique : que se passe-t-il de part et d'autre de $h\Omega = 2$ ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une hessienne, à la main et par différences centrées, avec un pas bien choisi ?
2. Sais-tu classer un point critique de $\mathbb{R}^n$ par les signes des valeurs propres de sa hessienne ?
3. Sais-tu dire si une itération linéaire converge, et combien d'itérations elle demande ?
4. Sais-tu choisir le pas de Richardson, prévoir l'effet du conditionnement, et arrêter l'itération au bon moment ?
5. Sais-tu décider de la convergence d'une récurrence d'ordre 2 par le module de ses racines ?

**La suite** : le chapitre 19 de workbookIA, qui règle et compare des méthodes de descente par la courbure, puis M20 (tenseurs, axes et estimateurs pondérés), juste avant le chapitre 20 de workbookIA.